# 01 — Exploratory Data Analysis

Credit-card **default** prediction (binary classification). Goal of this notebook:
understand the class imbalance, the feature distributions, the documented vs.
**undocumented** categorical codes, and missingness — everything that motivates the
cleaning in [`src/preprocessing.py`](../src/preprocessing.py).

> Run `main.py` for the actual pipeline; this notebook is analysis only.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # make `src` importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_development
from src import config

sns.set_theme(style='whitegrid')
df = load_development()
print(df.shape)
df.head()

## Class balance

The target is imbalanced (non-default dominates). This is why we optimise **macro-F1**
rather than accuracy.

In [ ]:
label = config.detect_label_col(df)
counts = df[label].value_counts().sort_index()
print(counts)
print((counts / counts.sum()).round(3))

ax = counts.plot(kind='bar', color=['#4c72b0', '#dd8452'])
ax.set_xticklabels(['no default (0)', 'default (1)'], rotation=0)
ax.set_title('Class balance')
plt.show()

## Categorical features & undocumented codes

The spec documents `EDUCATION` as 1..4 and `MARRIAGE` as 1..3, but the real data also
contains **undocumented** codes. We fold `EDUCATION {0,5,6} -> 4` and `MARRIAGE {0} -> 3`
in preprocessing.

In [ ]:
for col in config.CATEGORICAL:
    print(col, dict(df[col].value_counts().sort_index()))

## Repayment status (PAY_*)

Documented as -1 (paid duly) and 1..9 (months late), but the real data also contains
**-2 and 0**. We keep PAY_* as ordinal numeric.

In [ ]:
for col in config.PAY_COLS:
    print(col, sorted(df[col].unique()))

## Distributions of a few key numeric features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(df['LIMIT_BAL'], bins=40, ax=axes[0]); axes[0].set_title('LIMIT_BAL')
sns.histplot(df['AGE'], bins=40, ax=axes[1]); axes[1].set_title('AGE')
sns.countplot(x='PAY_0', data=df, ax=axes[2]); axes[2].set_title('PAY_0')
plt.tight_layout(); plt.show()

## Missingness

The dataset is normally complete, but we impute defensively in the pipeline anyway.

In [ ]:
na = df.isna().sum()
print(na[na > 0] if na.any() else 'No missing values.')

## Correlation heatmap (numeric features)

In [ ]:
num = df[config.NUMERIC + [label]]
plt.figure(figsize=(11, 9))
sns.heatmap(num.corr(), cmap='coolwarm', center=0, square=True)
plt.title('Correlation (numeric features + target)')
plt.show()

## Takeaways

- Strong imbalance -> optimise **macro-F1**, tune the decision threshold.
- Fold undocumented `EDUCATION`/`MARRIAGE` codes (done in `src/preprocessing.py`).
- `PAY_*` carry the strongest signal; kept as ordinal numeric.
- No missing values in practice, but the pipeline imputes defensively.

Next: see [`src/models.py`](../src/models.py) and run `python main.py`.